# 17.5 一組 scale 應該管多大範圍？

# 第 17 章：量化，用更少位元表示模型

前置：第2章Linear與數值。像把連續刻度尺改成較粗的格子：先看誤差，再看真正的儲存bytes，最後才談硬體加速。quantization.py的參考Linear會反量化，不會假裝是低位元kernel。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：換較粗刻度尺：省格子但有誤差**

縮小儲存與硬體加速分開驗證。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/quantization.svg")))

**先想一想：**一行很大、一行很小，共用scale會損害誰？

一把scale管整張表，極值會牽連其他行。每row或每group各用刻度能降低部分誤差，但要存更多metadata。

**把直覺寫成數學：**per-output-channel scale:[out,1]；per-group還需保存分組尺寸與尾塊規則。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.quantization import quantize_symmetric

w = torch.tensor([[0.01, 0.03, 0.05, 0.08], [10.0, 30.0, 50.0, 80.0]])
for channel in (False, True):
    q, s = quantize_symmetric(w, 4, per_channel=channel)
    print(channel, "每行MAE", (w - q * s).abs().mean(-1), "scale格數", s.numel())

**如何讀結果：**per-channel以output row爲單位；per-group可把同一row再分更短組，需另外比較metadata。

**只改一件事：**只把大行極值減小。
